# FlyFFN vs SmolLM2-135M — isolated FFN replacement

This experiment keeps **standard SmolLM2 attention unchanged** and replaces only the FFN/MLP blocks with a FlyWire-routed sparse SwiGLU design.

Each pretrained FFN is split exactly into 8 disjoint shards. Before sparse activation, the shards sum exactly to the original FFN. Layers are then progressively calibrated into **Top-2-of-8 FlyFFN** routing using the real FlyWire topology, with a rewired control.

The first goal is quality: compare CE/PPL and generated text against stock SmolLM2 while attention remains untouched. The current prototype computes all shard outputs before Top-k selection, so measured speed is **not yet the final sparse-kernel speed**; theoretical active FFN fraction is 25%.

In [ ]:
#@title 1. Update repository and install
import pathlib, subprocess, sys
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),
                'transformers>=4.56','datasets>=3.0','accelerate>=1.0',
                'huggingface_hub>=0.34','pandas>=2.0','requests>=2.31','tqdm>=4.66'], check=True)
print('Ready:', REPO_DIR)


In [ ]:
#@title 2. FlyFFN configuration
RUN_MODE = 'quick' #@param ['quick','strong']
SEQ_LEN = 128 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
FLY_NODES = 256 #@param {type:'integer'}
ROUTER_RANK = 64 #@param {type:'integer'}
MAX_EDGES = 2048 #@param {type:'integer'}
NUM_SHARDS = 8 #@param {type:'integer'}
TOP_K = 2 #@param {type:'integer'}
GRAPH_STEPS = 1 #@param {type:'integer'}
GRAPH_MIX_INIT = 0.50 #@param {type:'number'}
RUN_REWIRED_CONTROL = True #@param {type:'boolean'}
OUTPUT_DIR = REPO_DIR / 'results' / 'flyffn_smollm2_135m'
print('Attention: stock SmolLM2')
print(f'FlyFFN: Top-{TOP_K}-of-{NUM_SHARDS} | theoretical active FFN fraction = {TOP_K/NUM_SHARDS:.1%}')
print('Output:', OUTPUT_DIR)


In [ ]:
#@title 3. Train/evaluate FlyFFN — live output + status + ETA
import os, re, time, queue, threading, subprocess
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_smollm2_flyffn.py'),
     '--run-mode',RUN_MODE,'--seq-len',str(SEQ_LEN),'--batch-size',str(BATCH_SIZE),
     '--fly-nodes',str(FLY_NODES),'--router-rank',str(ROUTER_RANK),'--max-edges',str(MAX_EDGES),
     '--num-shards',str(NUM_SHARDS),'--top-k',str(TOP_K),'--graph-steps',str(GRAPH_STEPS),
     '--graph-mix-init',str(GRAPH_MIX_INIT),'--output-dir',str(OUTPUT_DIR)]
if RUN_REWIRED_CONTROL: cmd.append('--rewired')
UPDATES = 300 if RUN_MODE == 'quick' else 2500
MODELS = 2 if RUN_REWIRED_CONTROL else 1
TOTAL = UPDATES * MODELS
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['TQDM_MININTERVAL']='1'
print('='*92)
print('FlyFFN / SmolLM2-135M isolated FFN experiment')
print(f'Mode={RUN_MODE} | standard attention unchanged | Top-{TOP_K}-of-{NUM_SHARDS} FlyFFN')
print('Command:', ' '.join(cmd))
print('='*92, flush=True)
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
q=queue.Queue(); DONE=object()
def reader():
    try:
        for line in iter(p.stdout.readline,''): q.put(line.rstrip('\n'))
    finally: q.put(DONE)
threading.Thread(target=reader,daemon=True).start()
gre=re.compile(r'^GLOBAL\s+(biological|rewired)\s+(\d+)/(\d+).*?ups=([0-9.]+)\s+eta_s=([0-9.]+)')
cre=re.compile(r'^CALIB\s+(biological|rewired)\s+group=(\d+)/(\d+).*?step=(\d+)/(\d+)')
done={'biological':0,'rewired':0}; stage='setup'; eta=None; start=time.time(); last=0; finished=False
while True:
    now=time.time()
    try: item=q.get(timeout=1)
    except queue.Empty: item=None
    if item is DONE: finished=True
    elif item:
        print(item, flush=True)
        if item.startswith('STAGE '): stage=item[6:]
        m=cre.search(item)
        if m: stage=f'FFN calibration {m.group(1)} group {m.group(2)}/{m.group(3)} step {m.group(4)}/{m.group(5)}'
        m=gre.search(item)
        if m:
            stage=f'global distillation {m.group(1)}'; done[m.group(1)]=int(m.group(2)); eta=float(m.group(5))
    if now-last>=10:
        n=done['biological']+(done['rewired'] if RUN_REWIRED_CONTROL else 0); frac=n/max(TOTAL,1)
        bar='#'*int(frac*28)+'-'*(28-int(frac*28))
        def ft(s):
            if s is None: return 'calculating...'
            s=int(s); return f'{s//60}m {s%60:02d}s'
        print(f'\nSTATUS | {stage}\n[{bar}] {100*frac:5.1f}% ({n}/{TOTAL}) | elapsed {ft(now-start)} | ETA {ft(eta)}\n', flush=True)
        last=now
    if p.poll() is not None and finished and q.empty(): break
rc=p.wait(); print('Finished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Results
import json, pandas as pd
from IPython.display import display
summary = pd.read_csv(OUTPUT_DIR/'summary.csv', index_col=0)
report = json.loads((OUTPUT_DIR/'report.json').read_text())
samples = json.loads((OUTPUT_DIR/'samples.json').read_text())
display(summary)
print('\nCHECKS')
print('Architecture:', report['architecture'])
print('Attention unchanged:', report['attention_unchanged'])
print('FlyFFN layers:', report['flyffn_layers'])
print('Theoretical active FFN fraction:', report['theoretical_active_ffn_fraction'])
print('Device:', report['device'], '| dtype:', report['dtype'])
print('\nKEY METRICS')
for k in ['fly_ce_gap_vs_smollm2','fly_ppl_ratio_vs_smollm2','parameter_ratio_fly_over_smollm2',
          'decode_speed_ratio_fly_over_smollm2','biological_topology_ce_gain','biological_topology_ppl_gain_pct']:
    if k in report: print(k, ':', report[k])
print('\nImportant: this v1 quality prototype still computes all 8 shard outputs before Top-k gather.')
print('A fused sparse dispatch kernel is needed before interpreting runtime speed as the final FlyFFN speed.')
print('\nSAMPLES')
for x in samples:
    print('='*90); print('PROMPT:',x['prompt']); print(x['text'])
